# 🎬 Auto Video Studio — Formula DNA + OmniVoice + Pexels

**Run cells 1 → 2 → 3 in order.**  
Cell 3 launches the full Gradio UI with a public URL.

**Before running:**  
- Runtime → Change runtime type → **T4 GPU**  
- Get a free Gemini API key at [aistudio.google.com](https://aistudio.google.com)  
- Get a free Pexels API key at [pexels.com/api](https://www.pexels.com/api/)  
- Have your Writing Formula ready (same formula from your video editor system)  
- Have a 3–10s reference voice audio clip (.wav or .mp3)

## Cell 1 — Install Dependencies

In [ ]:
import subprocess, sys, os

print('📦 Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'omnivoice', 'transformers>=5.3', 'gradio==6.11.0',
    'google-generativeai', 'pydub', 'requests', 'scipy', 'soundfile', 'numpy',
], check=True)

result = subprocess.run(['ffmpeg', '-version'], capture_output=True)
if result.returncode == 0:
    print('✅ ffmpeg found')
else:
    subprocess.run(['apt-get', 'install', '-y', '-q', 'ffmpeg'], check=True)
    print('✅ ffmpeg installed')

PROJ = '/content/video_project'
for d in ['audio', 'stock_videos', 'assets', 'subtitles', 'output', 'cache']:
    os.makedirs(f'{PROJ}/{d}', exist_ok=True)

print('✅ All dependencies installed')
print('✅ Project directories created at', PROJ)
print('\n➡️  Run Cell 2 to load OmniVoice model')


## Cell 2 — Load OmniVoice Model  *(T4 GPU required)*

In [ ]:
import torch, numpy as np

print('🔊 Loading OmniVoice model on T4 GPU...')
print(f'   CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from omnivoice import OmniVoice, OmniVoiceGenerationConfig

OMNIVOICE_MODEL = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice',
    device_map='cuda',
    dtype=torch.float16,
    load_asr=True,
    token=False,
)
SAMPLING_RATE = OMNIVOICE_MODEL.sampling_rate

print(f'✅ OmniVoice loaded — sampling rate: {SAMPLING_RATE} Hz')
print('\n➡️  Run Cell 3 to launch the Gradio UI')


## Cell 3 — Pipeline Functions + Gradio UI

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  IMPORTS & SETUP
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, re, json, time, hashlib, shutil, subprocess, traceback
from pathlib import Path
from typing import List, Dict, Optional, Tuple
import requests
import numpy as np
import soundfile as sf
import gradio as gr
import google.generativeai as genai

PROJ = '/content/video_project'
DNA_CACHE = Path(f'{PROJ}/cache')
DNA_CACHE.mkdir(exist_ok=True)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  FORMULA DNA HELPERS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _formula_key(formula):
    return hashlib.md5(formula.encode()).hexdigest()

def _load_dna(formula):
    p = DNA_CACHE / f'{_formula_key(formula)}.json'
    if p.exists():
        try:
            return json.loads(p.read_text()).get('dna', '')
        except Exception:
            pass
    return ''

def _save_dna(formula, dna):
    p = DNA_CACHE / f'{_formula_key(formula)}.json'
    p.write_text(json.dumps({'dna': dna, 'len': len(formula)}))

def _gemini_call(api_key, prompt, model='gemini-2.5-flash', temperature=0.7, max_tokens=32768):
    genai.configure(api_key=api_key)
    m = genai.GenerativeModel(model)
    resp = m.generate_content(
        prompt,
        generation_config={'temperature': temperature, 'max_output_tokens': max_tokens},
    )
    return resp.text.strip()

def _xml(tag, text, fallback=''):
    m = re.search(rf'<{tag}>(.*?)</{tag}>', text, re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else fallback

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  PHASE 1 — FORMULA DNA EXTRACTION
#  Same architecture as the production video editor system.
#  DNA cached to disk — same formula skips extraction on re-run.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def extract_formula_dna(api_key, formula, title, num_chunks, language='English'):
    cached_dna = _load_dna(formula)

    outline_fmt = ''
    for i in range(1, num_chunks + 1):
        role = 'HOOK/OPENING' if i == 1 else ('CLOSING/CTA' if i == num_chunks else 'BODY')
        closing = (
            'CLOSING RULES: Build toward CTA only. No new tangents. '
            'Deliver the title climax. Copy CTA phrase verbatim. Script feels 100% complete.\n'
            if i == num_chunks else ''
        )
        outline_fmt += (
            f'<chunk_{i}_outline>\n'
            f'STEP-BY-STEP recipe for chunk {i} ({role}) — 100% specific to "{title}".\n'
            f'Each step: STEP N | WHAT TO WRITE | FORMULA TECHNIQUE | MANDATORY PHRASE\n'
            f'{closing}'
            f'</chunk_{i}_outline>\n\n'
            f'<chunk_{i}_keyword>1-3 word Pexels stock video search keyword for chunk {i} visual scene</chunk_{i}_keyword>\n\n'
        )

    if cached_dna and len(cached_dna) >= 1500:
        prompt = (
            f'You are a script outline generator.\n'
            f'Use the Formula DNA below to create chunk-specific writing recipes for this title.\n\n'
            f'FORMULA DNA (all niche laws already extracted):\n{cached_dna}\n\n'
            f'VIDEO TITLE: "{title}"\nLANGUAGE: {language}\nCHUNKS: {num_chunks}\n\n'
            f'Output ONLY the XML below. No markdown, no code blocks.\n\n'
            f'<anchor>main subject at the heart of this video title</anchor>\n'
            f'<cta>exact CTA text from the DNA verbatim</cta>\n\n'
            f'{outline_fmt}'
        )
        text = _gemini_call(api_key, prompt, temperature=0.1, max_tokens=16384)
        dna = cached_dna
        print('   ✅ Formula DNA loaded from disk cache (fast path)')
    else:
        dna_template = (
            '## 1. OPENING LAWS\n[Copy exact opening phrases/hooks verbatim.]\n\n'
            '## 2. TONE & VOICE LAWS\n[Exact tone, register, energy, person.]\n\n'
            '## 3. STRUCTURE LAWS\n[Paragraph/sentence rules. Section lengths. Transitions.]\n\n'
            '## 4. MANDATORY PHRASES & HOOKS\n[EVERY phrase that MUST appear — copy verbatim.]\n\n'
            '## 5. FORBIDDEN WORDS & PATTERNS\n[Every banned word/technique — copy exactly.]\n\n'
            '## 6. STORYTELLING & CONTENT LAWS\n[Story structure, examples, social proof, data.]\n\n'
            '## 7. ENGAGEMENT & RETENTION LAWS\n[Hooks, cliffhangers, pattern interrupts.]\n\n'
            '## 8. PROMO & CTA LAWS\n[Promo count, placement, exact CTA verbatim.]\n\n'
            '## 9. LANGUAGE & STYLE LAWS\n[Vocabulary level, contractions, idioms.]\n\n'
            '## 10. NICHE-SPECIFIC LAWS\n[All domain-specific rules unique to this formula.]'
        )
        prompt = (
            f'You are an elite YouTube script production analyst.\n'
            f'Do TWO things:\n'
            f'1. EXTRACT COMPLETE FORMULA DNA from the Writing Guidelines — every rule, every law.\n'
            f'2. CREATE PRESCRIPTIVE SCRIPT OUTLINES — step-by-step recipe per chunk, specific to this title.\n\n'
            f'VIDEO TITLE: "{title}"\nLANGUAGE: {language}\nCHUNKS: {num_chunks}\n\n'
            f'<writing_guidelines>\n{formula}\n</writing_guidelines>\n\n'
            f'Output ONLY the XML below. No markdown, no code blocks.\n\n'
            f'<anchor>main subject at the heart of this video title</anchor>\n'
            f'<cta>exact CTA text from Writing Guidelines verbatim</cta>\n\n'
            f'<formula_dna>\n'
            f'CRITICAL: Extract EVERY law. Be EXHAUSTIVE. Miss nothing.\n\n'
            f'{dna_template}\n'
            f'</formula_dna>\n\n'
            f'{outline_fmt}'
        )
        print('   📤 Extracting formula DNA + outlines (first-time, ~20-30s)...')
        text = _gemini_call(api_key, prompt, temperature=0.1, max_tokens=65536)
        raw_dna = _xml('formula_dna', text)
        dna = raw_dna if len(raw_dna) >= 1500 else formula[:12000]
        if len(dna) >= 1500:
            _save_dna(formula, dna)
            print(f'   ✅ Formula DNA extracted ({len(dna):,} chars) — cached to disk')

    outlines, keywords = {}, {}
    for i in range(1, num_chunks + 1):
        m = re.search(rf'<chunk_{i}_outline>(.*?)</chunk_{i}_outline>', text, re.DOTALL | re.IGNORECASE)
        if m and len(m.group(1).strip()) > 50:
            outlines[i] = m.group(1).strip()
        kw = re.search(rf'<chunk_{i}_keyword>(.*?)</chunk_{i}_keyword>', text, re.DOTALL | re.IGNORECASE)
        keywords[i] = kw.group(1).strip() if kw else 'finance trading'

    return {
        'dna'     : dna,
        'outlines': outlines,
        'keywords': keywords,
        'anchor'  : _xml('anchor', text, title),
        'cta'     : _xml('cta', text, 'Subscribe and hit the bell'),
    }

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  PHASE 2 — WRITE ONE CHUNK
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def write_chunk(api_key, title, language, formula, plan,
                chunk_idx, total_chunks, target_chars, previous_context=''):
    dna      = plan['dna']
    outline  = plan['outlines'].get(chunk_idx, f'Write part {chunk_idx} of the video about "{title}".')
    is_hook  = chunk_idx == 1
    is_final = chunk_idx == total_chunks
    role     = 'HOOK / OPENING' if is_hook else ('CLOSING / CTA' if is_final else 'BODY / DEVELOPMENT')

    formula_block = f'\n<raw_writing_guidelines>\n{formula}\n</raw_writing_guidelines>\n' if len(formula) <= 65000 else ''
    continuation  = (f'\nCONTINUATION — last 500 chars of previous chunk (continue naturally):\n"{previous_context}"\n'
                     if previous_context else '')

    hook_check  = 'Hook grabs attention in first 3 seconds' if is_hook else ''
    close_check = 'CTA placed exactly as formula requires' if is_final else ''
    body_check  = 'Smooth natural continuation from previous chunk' if not is_hook and not is_final else ''
    flow_check  = hook_check or close_check or body_check

    system = (
        'You are an elite YouTube scriptwriter. '
        'Execute the writing recipe step by step. '
        'Every sentence must execute a specific formula rule. '
        'Output ONLY the script text — no commentary, no headings, no chunk labels.'
    )

    prompt = (
        f'FORMULA DNA (ALL NICHE LAWS):\n{dna}\n'
        f'{formula_block}'
        f'\nVIDEO TITLE: "{title}"\nLANGUAGE: {language}\n'
        f'CHUNK: {chunk_idx} of {total_chunks} ({role})\n'
        f'TARGET LENGTH: {target_chars:,} characters\n'
        f'{continuation}'
        f'\nSTEP-BY-STEP RECIPE:\n{outline}\n\n'
        f'══ SELF-CHECK ══\n'
        f'✓ Every sentence executes a formula rule\n'
        f'✓ Mandatory phrases used verbatim\n'
        f'✓ No forbidden words/patterns\n'
        f'✓ Tone matches formula exactly\n'
        f'✓ {flow_check}\n'
        f'✓ Length ~{target_chars:,} characters\n'
        f'═══════════════\n\n'
        f'Write chunk {chunk_idx} script text NOW:'
    )

    max_tokens = min(65536, max(int(target_chars / 2) + 6000, 12000))
    temp = 0.3 if is_hook else (0.2 if is_final else 0.7)

    genai.configure(api_key=api_key)
    m = genai.GenerativeModel('gemini-2.5-flash', system_instruction=system)
    resp = m.generate_content(
        prompt,
        generation_config={'temperature': temp, 'max_output_tokens': max_tokens},
    )
    return resp.text.strip()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  FULL SCRIPT GENERATION
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def generate_full_script(api_key, title, formula, num_chunks,
                         language='English', chars_per_chunk=1100, progress=None):
    if progress:
        progress(0.05, desc='Phase 1: Extracting formula DNA...')

    plan = extract_formula_dna(api_key, formula, title, num_chunks, language)
    print(f'\n✅ Phase 1 done — anchor: "{plan["anchor"]}"')

    chunks_out, prev_ctx = [], ''

    for i in range(1, num_chunks + 1):
        if progress:
            progress(0.1 + (i / num_chunks) * 0.85, desc=f'Phase 2: Writing chunk {i}/{num_chunks}...')
        print(f'\n✍️  Writing chunk {i}/{num_chunks}...')

        text = write_chunk(
            api_key=api_key, title=title, language=language, formula=formula,
            plan=plan, chunk_idx=i, total_chunks=num_chunks,
            target_chars=chars_per_chunk, previous_context=prev_ctx,
        )
        text = re.sub(r'=== END.*', '', text, flags=re.IGNORECASE).strip()

        keyword = plan['keywords'].get(i, 'finance trading')
        print(f'   ✅ {len(text):,} chars — keyword: "{keyword}"')

        chunks_out.append({'chunk_id': i, 'text': text, 'keyword': keyword})
        prev_ctx = text[-500:]
        time.sleep(0.3)

    total = sum(len(c['text']) for c in chunks_out)
    print(f'\n🎉 Script done: {len(chunks_out)} chunks, {total:,} chars (~{total//5} words)')
    return chunks_out

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  VOICE CLONE — SAVE / LOAD / REUSE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

VOICE_PROFILE_PATH = f'{PROJ}/assets/voice_profile.json'
_voice_clone_prompt_cache = None   # survives all Gradio calls in session

def clone_and_save_voice(ref_audio_path):
    global _voice_clone_prompt_cache
    if not ref_audio_path or not os.path.exists(ref_audio_path):
        return '❌ No audio file provided'
    ref_dest = f'{PROJ}/assets/voice_reference.wav'
    shutil.copy2(ref_audio_path, ref_dest)
    _voice_clone_prompt_cache = OMNIVOICE_MODEL.create_voice_clone_prompt(ref_audio=ref_dest)
    with open(VOICE_PROFILE_PATH, 'w') as f:
        json.dump({'ref_audio': ref_dest}, f)
    return '✅ Voice cloned and saved — reused for all audio generation automatically'

def _ensure_voice_loaded():
    global _voice_clone_prompt_cache
    if _voice_clone_prompt_cache is not None:
        return True
    if os.path.exists(VOICE_PROFILE_PATH):
        with open(VOICE_PROFILE_PATH) as f:
            src = json.load(f).get('ref_audio', '')
        if src and os.path.exists(src):
            _voice_clone_prompt_cache = OMNIVOICE_MODEL.create_voice_clone_prompt(ref_audio=src)
            print('✅ Voice profile reloaded from saved reference')
            return True
    return False

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  TTS — GENERATE AUDIO PER CHUNK
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def generate_chunk_audio(chunk_text, chunk_idx, language='Auto'):
    if not _ensure_voice_loaded():
        raise ValueError('Voice not cloned — clone a voice first in Settings tab')

    lang_map = {'Auto': 'auto', 'English': 'en', 'French': 'fr', 'Spanish': 'es',
                'German': 'de', 'Arabic': 'ar', 'Portuguese': 'pt', 'Italian': 'it'}
    lang_code = lang_map.get(language, 'auto')

    gen_config = OmniVoiceGenerationConfig(
        num_step=32, guidance_scale=3.0, denoise=0.8,
        preprocess_prompt=True, postprocess_output=True,
    )

    audio = OMNIVOICE_MODEL.generate(
        text=chunk_text, language=lang_code,
        generation_config=gen_config,
        voice_clone_prompt=_voice_clone_prompt_cache,
    )

    out_path = f'{PROJ}/audio/chunk_{chunk_idx:02d}.wav'
    sf.write(out_path, audio, SAMPLING_RATE)
    duration = len(audio) / SAMPLING_RATE
    print(f'   🎤 Chunk {chunk_idx}: {duration:.1f}s')
    return out_path, duration

def generate_all_audio(chunks, language='Auto', progress=None):
    results = []
    for i, chunk in enumerate(chunks):
        if progress:
            progress(i / len(chunks), desc=f'TTS chunk {i+1}/{len(chunks)}...')
        path, dur = generate_chunk_audio(chunk['text'], chunk['chunk_id'], language)
        results.append({**chunk, 'audio_path': path, 'duration': dur})
    return results

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  WHISPER — WORD-LEVEL TIMESTAMPS (for karaoke captions)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def get_word_timestamps(audio_path, text):
    try:
        result = OMNIVOICE_MODEL.transcribe(audio_path, word_timestamps=True)
        word_times = []
        for seg in result.get('segments', []):
            for w in seg.get('words', []):
                word_times.append({'word': w['word'].strip(), 'start': w['start'], 'end': w['end']})
        if word_times:
            return word_times
    except Exception as e:
        print(f'   ASR fallback (uniform): {e}')

    words = text.split()
    data, sr = sf.read(audio_path)
    total_dur = len(data) / sr
    spw = total_dur / max(len(words), 1)
    return [{'word': w, 'start': i * spw, 'end': (i + 1) * spw} for i, w in enumerate(words)]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ASS SUBTITLE BUILDER — KARAOKE WORD HIGHLIGHTING
#  Current word: gold   Other words: bold white
#  Position: right half of 1920x1080 (MarginL=900 in ASS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def build_ass_subtitles(chunks_with_audio):
    ass_path = f'{PROJ}/subtitles/captions.ass'
    header = (
        '[Script Info]\n'
        'ScriptType: v4.00+\nWrapStyle: 0\nScaledBorderAndShadow: yes\n'
        'PlayResX: 1920\nPlayResY: 1080\n\n'
        '[V4+ Styles]\n'
        'Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, '
        'Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, '
        'Alignment, MarginL, MarginR, MarginV, Encoding\n'
        'Style: Default,Arial,54,&H00FFFFFF,&H000000FF,&H00000000,&H90000000,-1,0,0,0,100,100,0,0,1,3,2,'
        '5,900,80,120,1\n\n'
        '[Events]\n'
        'Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text\n'
    )

    def ts(sec):
        h = int(sec // 3600)
        m = int((sec % 3600) // 60)
        s = int(sec % 60)
        cs = int((sec % 1) * 100)
        return f'{h}:{m:02d}:{s:02d}.{cs:02d}'

    lines = []
    global_offset = 0.0

    for chunk in chunks_with_audio:
        audio_path = chunk.get('audio_path', '')
        duration   = chunk.get('duration', 0)
        text       = chunk.get('text', '')
        if not audio_path or not os.path.exists(audio_path):
            global_offset += duration
            continue

        wt = get_word_timestamps(audio_path, text)
        WORDS_PER_LINE = 7
        groups = [wt[i:i+WORDS_PER_LINE] for i in range(0, len(wt), WORDS_PER_LINE)]

        for group in groups:
            if not group:
                continue
            for wi, wdata in enumerate(group):
                w_start = wdata['start'] + global_offset
                w_end   = wdata['end']   + global_offset
                parts = []
                for j, wd in enumerate(group):
                    wclean = re.sub(r'[{}\\]', '', wd['word'])
                    if j == wi:
                        # Gold highlight: ASS colour &H00CFFF& (BGR = gold/yellow)
                        parts.append(r'{\c&H00CFFF&\b1}' + wclean + r'{\c&HFFFFFF&}')
                    else:
                        parts.append(wclean)
                lines.append(
                    f'Dialogue: 0,{ts(w_start)},{ts(w_end)},Default,,0,0,0,,{" ".join(parts)}'
                )
        global_offset += duration

    with open(ass_path, 'w', encoding='utf-8') as f:
        f.write(header + '\n'.join(lines))
    return ass_path

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  PEXELS — STOCK VIDEO DOWNLOAD
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def download_pexels_video(api_key, keyword, out_path):
    headers = {'Authorization': api_key}
    fallbacks = [keyword, 'finance trading', 'stock market chart', 'business money', 'abstract technology']

    for kw in fallbacks:
        try:
            url = (f'https://api.pexels.com/videos/search'
                   f'?query={kw}&orientation=landscape&per_page=10&size=large')
            resp = requests.get(url, headers=headers, timeout=20)
            if resp.status_code != 200:
                continue
            videos = resp.json().get('videos', [])
            if not videos:
                continue

            best, best_score = None, -1
            for vid in videos:
                for vf in vid.get('video_files', []):
                    w, h = vf.get('width', 0), vf.get('height', 0)
                    if w < 1280 or h < 720:
                        continue
                    score = w * h + (500000 if abs(w - 1920) < 200 else 0)
                    if vf.get('quality') in ('hd', 'full_hd'):
                        score += 300000
                    if score > best_score:
                        best_score = score
                        best = vf

            if not best:
                continue

            dl = requests.get(best['link'], stream=True, timeout=120)
            with open(out_path, 'wb') as f:
                for chunk_data in dl.iter_content(chunk_size=65536):
                    f.write(chunk_data)
            print(f'   📹 "{kw}" → {best.get("width")}x{best.get("height")}')
            return out_path

        except Exception as e:
            print(f'   Pexels error "{kw}": {e}')

    print(f'   ⚠️  No video for "{keyword}" (all fallbacks failed)')
    return None

def download_all_stock_videos(pexels_key, chunks, progress=None):
    results = []
    for i, chunk in enumerate(chunks):
        if progress:
            progress(i / len(chunks), desc=f'Video {i+1}/{len(chunks)}: "{chunk["keyword"]}"...')
        out_path = f'{PROJ}/stock_videos/stock_{chunk["chunk_id"]:02d}.mp4'
        path = download_pexels_video(pexels_key, chunk['keyword'], out_path)
        results.append({**chunk, 'video_path': path or ''})
        time.sleep(0.6)
    return results

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  FFMPEG — VIDEO RENDERER
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _vid_duration(path):
    try:
        r = subprocess.run(
            ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
             '-of', 'default=noprint_wrappers=1:nokey=1', path],
            capture_output=True, text=True)
        return float(r.stdout.strip())
    except Exception:
        return 10.0


def _render_segment(chunk, seg_out, ass_path, character_image):
    audio_path = chunk.get('audio_path', '')
    video_path = chunk.get('video_path', '')
    duration   = chunk.get('duration', 5.0)
    idx        = chunk.get('chunk_id', 0)

    if not audio_path or not os.path.exists(audio_path):
        print(f'  ⚠️  Chunk {idx}: no audio — skipping')
        return False

    has_char  = bool(character_image and os.path.exists(character_image))
    has_video = bool(video_path and os.path.exists(video_path))
    loops     = max(1, int(np.ceil(duration / max(_vid_duration(video_path), 0.1)))) if has_video else 1

    if has_video:
        bg = (f'[0:v]scale=1920:1080:force_original_aspect_ratio=increase,'
              f'crop=1920:1080,trim=duration={duration},setpts=PTS-STARTPTS[bg];'
              f'color=black:s=1920x1080:r=30[blk];'
              f'[bg][blk]blend=all_mode=normal:all_opacity=0.55[darkbg]')
        vid_args = ['-stream_loop', str(loops - 1), '-i', video_path]
    else:
        bg = f'color=black:s=1920x1080:r=30:d={duration}[darkbg]'
        vid_args = ['-f', 'lavfi', '-i', f'color=black:s=1920x1080:r=30:d={duration}']

    char_args  = ['-i', character_image] if has_char else []
    audio_idx  = (1 if not has_video else 1) + (1 if has_char else 0)

    if has_char:
        char_in = 1 if has_video else 1
        vf = (f'{bg};'
              f'[{char_in}:v]scale=-1:518:flags=lanczos[char];'
              f'[darkbg][char]overlay=x=30:y=(H-h)/2+60[withchar];'
              f'[withchar]ass={ass_path}[out]')
    else:
        vf = f'{bg};[darkbg]ass={ass_path}[out]'

    cmd = (
        ['ffmpeg', '-y']
        + vid_args + char_args
        + ['-i', audio_path]
        + ['-filter_complex', vf]
        + ['-map', '[out]', '-map', f'{audio_idx}:a']
        + ['-t', str(duration)]
        + ['-c:v', 'libx264', '-preset', 'fast', '-crf', '20', '-b:v', '5000k']
        + ['-c:a', 'aac', '-ar', '44100', '-b:a', '192k']
        + ['-r', '30', '-pix_fmt', 'yuv420p', seg_out]
    )

    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  ⚠️  Segment {idx} error:\n{r.stderr[-400:]}')
        return False
    print(f'  ✅ Segment {idx}: {duration:.1f}s')
    return True


def render_final_video(chunks, character_image=None, bg_music=None, progress=None):
    if progress:
        progress(0.03, desc='Building karaoke captions...')
    ass_path = build_ass_subtitles(chunks)
    print(f'✅ ASS: {ass_path}')

    segment_paths = []
    for i, chunk in enumerate(chunks):
        if progress:
            progress(0.08 + (i / len(chunks)) * 0.72, desc=f'Segment {i+1}/{len(chunks)}...')
        seg_out = f'{PROJ}/output/segment_{i+1:02d}.mp4'
        if _render_segment(chunk, seg_out, ass_path, character_image):
            segment_paths.append(seg_out)

    if not segment_paths:
        raise ValueError('No segments rendered — check audio and stock video steps')

    if progress:
        progress(0.83, desc='Concatenating...')

    concat_list = f'{PROJ}/output/concat.txt'
    with open(concat_list, 'w') as f:
        for p in segment_paths:
            f.write(f"file '{p}'\n")

    joined = f'{PROJ}/output/joined.mp4'
    subprocess.run(['ffmpeg', '-y', '-f', 'concat', '-safe', '0',
                    '-i', concat_list, '-c', 'copy', joined],
                   check=True, capture_output=True)

    final_path = f'{PROJ}/output/final_video.mp4'

    if bg_music and os.path.exists(bg_music):
        if progress:
            progress(0.92, desc='Mixing background music...')
        subprocess.run([
            'ffmpeg', '-y', '-i', joined, '-stream_loop', '-1', '-i', bg_music,
            '-filter_complex',
            '[1:a]volume=0.12[bgm];[0:a][bgm]amix=inputs=2:duration=first:dropout_transition=3[outa]',
            '-map', '0:v', '-map', '[outa]', '-c:v', 'copy', '-c:a', 'aac', '-b:a', '192k', final_path,
        ], check=True, capture_output=True)
    else:
        shutil.copy2(joined, final_path)

    if progress:
        progress(1.0, desc='Done!')

    size_mb = os.path.getsize(final_path) / 1e6
    print(f'\n🎉 Final video: {final_path} ({size_mb:.1f} MB)')
    return final_path

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  SESSION STATE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

_session = {'chunks': [], 'character_img': None, 'bg_music': None}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  GRADIO HANDLERS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def handle_clone_voice(ref_audio):
    if ref_audio is None:
        return '❌ Upload a voice reference audio file first'
    return clone_and_save_voice(ref_audio)

def handle_generate_script(gemini_key, pexels_key, formula, num_chunks,
                            chars_per_chunk, title, language, progress=gr.Progress()):
    if not gemini_key.strip():
        return None, '❌ Gemini API key required'
    if not formula.strip() or len(formula.strip()) < 100:
        return None, '❌ Paste your writing formula (minimum 100 chars)'
    if not title.strip():
        return None, '❌ Enter a video title'
    try:
        chunks = generate_full_script(
            api_key=gemini_key.strip(), title=title.strip(), formula=formula.strip(),
            num_chunks=int(num_chunks), language=language,
            chars_per_chunk=int(chars_per_chunk), progress=progress,
        )
        _session['chunks'] = chunks
        table = [[c['chunk_id'], c['keyword'],
                  c['text'][:180] + '...' if len(c['text']) > 180 else c['text']]
                 for c in chunks]
        total_chars = sum(len(c['text']) for c in chunks)
        status = (f'✅ Script: {len(chunks)} chunks, {total_chars:,} chars '
                  f'(~{total_chars//5} words, ~{total_chars//900} min video)')
        return table, status
    except Exception as e:
        return None, f'❌ {e}\n{traceback.format_exc()[-500:]}'

def handle_generate_audio(language, progress=gr.Progress()):
    if not _session['chunks']:
        return '❌ Generate a script first (Step 1)'
    if not _ensure_voice_loaded():
        return '❌ Clone a voice first in the Settings tab'
    try:
        updated = generate_all_audio(_session['chunks'], language, progress)
        _session['chunks'] = updated
        total_dur = sum(c.get('duration', 0) for c in updated)
        return f'✅ Audio: {len(updated)} chunks, {total_dur:.1f}s (~{total_dur/60:.1f} min)'
    except Exception as e:
        return f'❌ {e}'

def handle_download_videos(pexels_key, progress=gr.Progress()):
    if not _session['chunks']:
        return '❌ Generate a script first'
    if not pexels_key.strip():
        return '❌ Pexels API key required'
    try:
        updated = download_all_stock_videos(pexels_key.strip(), _session['chunks'], progress)
        _session['chunks'] = updated
        found = sum(1 for c in updated if c.get('video_path'))
        return f'✅ Stock videos: {found}/{len(updated)} downloaded'
    except Exception as e:
        return f'❌ {e}'

def handle_set_character(img_path):
    if img_path:
        dest = f'{PROJ}/assets/character.png'
        shutil.copy2(img_path, dest)
        _session['character_img'] = dest
        return '✅ Character image saved'
    return ''

def handle_set_bgmusic(audio_path):
    if audio_path:
        dest = f'{PROJ}/assets/bg_music.mp3'
        shutil.copy2(audio_path, dest)
        _session['bg_music'] = dest
        return '✅ Background music saved'
    return ''

def handle_render_video(progress=gr.Progress()):
    if not _session['chunks']:
        return None, '❌ Complete Steps 1–3 first'
    missing = [c['chunk_id'] for c in _session['chunks'] if not c.get('audio_path')]
    if missing:
        return None, f'❌ Missing audio for chunks: {missing} — run Step 2'
    try:
        path = render_final_video(
            chunks=_session['chunks'],
            character_image=_session.get('character_img'),
            bg_music=_session.get('bg_music'),
            progress=progress,
        )
        return path, f'✅ Final video: {os.path.getsize(path)/1e6:.1f} MB'
    except Exception as e:
        return None, f'❌ {e}\n{traceback.format_exc()[-600:]}'

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  GRADIO UI
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

CSS = """
.step-label { font-size:17px; font-weight:700; color:#c4b5fd;
              border-left:4px solid #7c3aed; padding-left:10px; margin:10px 0 6px; }
footer { display:none !important; }
"""

with gr.Blocks(title='🎬 Auto Video Studio', theme=gr.themes.Soft(), css=CSS) as demo:

    gr.Markdown(
        '# 🎬 Auto Video Studio\n'
        '**Formula DNA Script** → **OmniVoice TTS** → **Pexels Stock Video** → '
        '**Karaoke Captions** → **Final MP4**'
    )

    with gr.Tabs():

        with gr.Tab('⚙️  Settings'):
            gr.Markdown('### 🔑 API Keys')
            with gr.Row():
                gemini_key = gr.Textbox(label='Gemini API Key',
                                        placeholder='AIza...', type='password', scale=1)
                pexels_key = gr.Textbox(label='Pexels API Key',
                                        placeholder='Pexels API key...', type='password', scale=1)

            gr.Markdown('### ✍️  Writing Formula  *(same formula as your video editor system)*')
            formula_box = gr.Textbox(
                label='Writing Formula / Writing Guidelines',
                placeholder=(
                    'Paste your complete writing formula here...\n\n'
                    'Same formula/writing guidelines you use in your video editor system.\n'
                    'Any size — 1KB to 300KB.\n\n'
                    'Phase 1: Extracts all laws (tone, hooks, structure, forbidden words, CTA)\n'
                    'Phase 2: Writes each chunk using the DNA as law\n\n'
                    'DNA cached to disk — same formula on next run skips extraction (saves ~25s).'
                ),
                lines=15, max_lines=60,
            )
            with gr.Row():
                num_chunks = gr.Slider(
                    label='Script Chunks  (each chunk = one scene + one stock video)',
                    minimum=3, maximum=20, value=8, step=1,
                    info='8 chunks ≈ 8-10 min  |  12 chunks ≈ 12-15 min',
                )
                chars_per_chunk = gr.Slider(
                    label='Characters per Chunk',
                    minimum=400, maximum=3000, value=1100, step=100,
                    info='1100 chars ≈ 170 words ≈ ~65s narration',
                )

            gr.Markdown('---\n### 🎤  Voice Clone  *(one-time setup per session)*')
            gr.Markdown(
                'Upload a **3–10 second** clean clip of the target voice.  \n'
                'Cloned voice saved and reused automatically for all audio generation.'
            )
            with gr.Row():
                voice_ref = gr.Audio(label='Voice Reference Audio', type='filepath', scale=2)
                clone_btn = gr.Button('🎤  Clone & Save Voice', variant='primary', scale=1)
            clone_status = gr.Textbox(label='Voice Status', interactive=False, lines=1)
            clone_btn.click(handle_clone_voice, inputs=[voice_ref], outputs=[clone_status])

            gr.Markdown('---\n### 🎭  Character Image & Background Music  *(optional)*')
            with gr.Row():
                char_img     = gr.Image(label='Character Image (PNG transparent bg)',
                                        type='filepath', scale=1)
                bg_music_inp = gr.Audio(label='Background Music (optional)',
                                        type='filepath', scale=1)
            with gr.Row():
                char_status = gr.Textbox(label='', interactive=False, lines=1)
                bgm_status  = gr.Textbox(label='', interactive=False, lines=1)
            char_img.change(handle_set_character,   inputs=[char_img],     outputs=[char_status])
            bg_music_inp.change(handle_set_bgmusic, inputs=[bg_music_inp], outputs=[bgm_status])

        with gr.Tab('🎬  Studio'):

            gr.HTML('<div class="step-label">① Generate Script  (Formula DNA + Gemini)</div>')
            with gr.Row():
                title_input = gr.Textbox(
                    label='Video Title',
                    placeholder="e.g. Warren Buffett's #1 Rule for Building Wealth in 2025",
                    scale=3,
                )
                lang_select = gr.Dropdown(
                    label='Language',
                    choices=['Auto', 'English', 'French', 'Spanish', 'German',
                             'Arabic', 'Portuguese', 'Italian'],
                    value='Auto', scale=1,
                )
            btn_script    = gr.Button('🧠  Generate Script', variant='primary')
            status_script = gr.Textbox(label='Status', interactive=False, lines=1)
            script_table  = gr.Dataframe(
                headers=['#', 'Pexels Keyword', 'Script Preview'],
                datatype=['number', 'str', 'str'],
                label='Script Chunks (Gemini auto-generates Pexels keywords)',
                wrap=True, height=280,
            )
            btn_script.click(
                handle_generate_script,
                inputs=[gemini_key, pexels_key, formula_box, num_chunks,
                        chars_per_chunk, title_input, lang_select],
                outputs=[script_table, status_script],
            )

            gr.HTML('<hr style="margin:20px 0"/>')

            gr.HTML('<div class="step-label">② Generate Voice Audio  (OmniVoice + your cloned voice)</div>')
            btn_audio    = gr.Button('🎤  Generate All Audio', variant='primary')
            status_audio = gr.Textbox(label='Status', interactive=False, lines=1)
            btn_audio.click(handle_generate_audio, inputs=[lang_select], outputs=[status_audio])

            gr.HTML('<hr style="margin:20px 0"/>')

            gr.HTML('<div class="step-label">③ Download Stock Videos  (Pexels — Gemini-matched keywords)</div>')
            gr.Markdown(
                'One stock video per chunk, matched to the scene.  '
                'Each loops to match chunk audio duration exactly.  \n'
                'Fallback chain: *chunk keyword* → `finance trading` → `stock market` → `abstract technology`'
            )
            btn_videos    = gr.Button('📹  Download Stock Videos', variant='primary')
            status_videos = gr.Textbox(label='Status', interactive=False, lines=1)
            btn_videos.click(handle_download_videos, inputs=[pexels_key], outputs=[status_videos])

            gr.HTML('<hr style="margin:20px 0"/>')

            gr.HTML('<div class="step-label">④ Render Final Video  (FFmpeg — 1920×1080 30fps)</div>')
            gr.Markdown(
                '**Layout:**  '
                'Stock video background + 55% dark overlay · Character image left · '
                'Karaoke captions right (gold current word) · Optional background music 12%'
            )
            btn_render    = gr.Button('🎬  Render Final Video', variant='primary')
            status_render = gr.Textbox(label='Status', interactive=False, lines=2)
            output_video  = gr.Video(label='Final Video', height=460)
            btn_render.click(handle_render_video, outputs=[output_video, status_render])

        with gr.Tab('❓  Help'):
            gr.Markdown("""
## Quick Start

### Settings tab (do this first)

| Setting | Notes |
|---|---|
| Gemini API Key | Free at aistudio.google.com |
| Pexels API Key | Free at pexels.com/api |
| Writing Formula | Paste the same formula from your video editor system |
| Chunks | 8 = ~8-10 min video  ·  12 = ~12-15 min |
| Voice Clone | Upload 3-10s clean voice clip → Clone & Save |
| Character Image | PNG with transparent background works best |

### Studio tab — run steps in order

**Step 1 — Generate Script (Formula DNA Architecture)**
- Gemini reads your formula and extracts DNA: ALL laws (tone, hooks, structure, forbidden words, CTA)
- Then writes each chunk step-by-step using the DNA as law
- Gemini also generates Pexels search keywords for each chunk's visual scene
- DNA is cached — same formula on next video skips extraction (saves ~25s)

**Step 2 — Generate Audio**
- OmniVoice TTS with your cloned voice
- Each chunk generates individually (~30-60s per chunk on T4)

**Step 3 — Download Stock Videos**
- One Pexels video per chunk keyword
- Auto-falls back if no results found
- Videos loop to match exact audio duration

**Step 4 — Render Final Video**
- FFmpeg composites all layers into 1920x1080 30fps MP4
- Character on left, karaoke captions on right, dark overlay on stock video

### Formula DNA — Same System as Your Video Editor

```
PHASE 1 (Gemini, 1 call):
  Read full Writing Guidelines (any size)
  Extract formula_dna: all laws in 10 categories
  Build per-chunk step-by-step recipe for this title
  → DNA cached by MD5 — reuses on same formula next run

PHASE 2 (Gemini, 1 call per chunk):
  Formula DNA (primary law)
  + Raw formula (dual-inject when under 65K chars)
  + Step-by-step outline specific to title and chunk
  + Self-check gate verified before output
```
""")

demo.launch(share=True, debug=False)
print('\n🚀 Gradio UI is live!')
